# 04 - EPIC-LAB Preprocessing

This notebook loads EPIC-LAB CSVs and prepares knee angle trajectories.
Update the column name below to match your files.


In [7]:
import sys
from pathlib import Path
import yaml

ROOT = Path(r"e:/Optimal_Control/PINN/hjb_pinn_exoskeleton")
sys.path.append(str(ROOT))

from src.datasets import load_epic_dataset

cfg = yaml.safe_load((ROOT / "configs" / "knee_config.yaml").read_text())
base = cfg["data_paths"]["epic_lab_base"]

# TODO: set this to actual column name in your CSVs
ANGLE_COL = "knee_sagittal"

print("Base path:", base)


Base path: E:/Dissertation/EPIC_LAB_csv_all


In [8]:
try:
    samples = load_epic_dataset(base, ANGLE_COL)
    print(f"Loaded {len(samples)} files")
except Exception as e:
    print("EPIC loading failed:", e)


Loaded 73 files


In [9]:
from scipy.signal import find_peaks
from src.datasets import EpicSample
import numpy as np

# Gait-cycle segmentation by knee angle peaks
N_POINTS = 101  # resample each cycle to fixed length
MIN_PEAK_DISTANCE = None
PROMINENCE_FRAC = 0.1

epic_cycles = []

for s in samples:
    theta = s.theta
    theta = np.deg2rad(theta)
    if len(theta) < 10:
        continue

    prominence = PROMINENCE_FRAC * (theta.max() - theta.min() + 1e-6)
    distance = MIN_PEAK_DISTANCE
    if distance is None:
        distance = max(10, len(theta) // 50)

    peaks, _ = find_peaks(theta, distance=distance, prominence=prominence)
    if len(peaks) < 2:
        continue

    for i in range(len(peaks) - 1):
        i0, i1 = peaks[i], peaks[i + 1]
        if i1 - i0 < 5:
            continue

        seg_theta = theta[i0:i1]
        t_old = np.linspace(0.0, 1.0, len(seg_theta))
        t_new = np.linspace(0.0, 1.0, N_POINTS)
        theta_rs = np.interp(t_new, t_old, seg_theta)

        epic_cycles.append(
            EpicSample(theta=theta_rs.astype(np.float32),
                       time=t_new.astype(np.float32))
        )

print(f"Segmented {len(epic_cycles)} gait cycles")

# Overwrite samples with segmented cycles for saving
samples = epic_cycles


Segmented 2502 gait cycles


In [10]:
import pickle

if 'samples' in globals() and samples:
    out_path = ROOT / 'data' / 'epic_clean.pkl'
    with out_path.open('wb') as f:
        pickle.dump(samples, f)
    print(f"Saved: {out_path}")
else:
    print("No samples to save.")


Saved: e:\Optimal_Control\PINN\hjb_pinn_exoskeleton\data\epic_clean.pkl


## Next Steps
- Add gait-cycle segmentation.
- Normalize time to [0, 1] per cycle.
- Save cleaned data to `data/epic_clean.pkl`.
